# 4DFlowNet: Architecture Sweep with Mathematical Interpolation Baselines

This notebook demonstrates the reproduction of the **4DFlowNet** architecture using a physically realistic k-space MRI acquisition simulation, compared against traditional mathematical interpolation baselines (**Trilinear**, **Tricubic**, and **Sinc**).

#### The Research Context:
In the original 4DFlowNet paper (Ferdian et al.), the authors attempted to use **PixelShuffle (sub-pixel convolution)** for 3D super-resolution but rejected it because it caused severe checkerboard artifacts and poor convergence in 3D velocity space. As a result, they settled for a fixed **trilinear resize layer** inside the network, followed by residual refinement convolutions.

#### The Upgrade:
We implement the 3D sub-pixel convolution (`PixelShuffle3d`), which the paper abandoned due to convergence issues, by using modern deep learning optimization practices:
- **AdamW** optimizer for stable weight regularization.
- **Cosine Annealing** learning rate scheduler to resolve high-frequency details.
- **Gradient Clipping** (`max_norm=1.0`) to suppress the gradient explosions that trigger checkerboard instabilities.

## 0. Colab Setup

Set your runtime to **GPU** (Runtime -> Change runtime type -> GPU). If running in Colab, clone the repository and install requirements.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/eliasubz/4D-FlowNet.git
    %cd 4D-FlowNet
    !pip install -q -r requirements.txt
else:
    print("Running locally. Directory content:")
    !pwd

## 1. Imports and Config

In [ ]:
import math
import torch
import torch.nn as nn
import torch.fft
from torch.utils.data import DataLoader
import scipy.ndimage
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd

from src.dataset import SyntheticFlowDataset, upsample_lr
from src.model import FourDFlowNet
from src.losses import four_d_flow_loss
from src.metrics import endpoint_error, peak_velocity_error, flow_rate_error, divergence_l1
from src.visualize import make_interactive_3d_flow_figure

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds for reproducible runs
torch.manual_seed(42)
np.random.seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)

## 2. Preview K-Space MRI Downsampling & 3D Geometry

Verify the physics-based k-space simulation. The pipeline applies VENC phase encoding, transforms to frequency domain (FFT), center-crops the k-space (downsampling), injects complex Gaussian noise, and returns magnitude and phase components.

In [ ]:
dataset = SyntheticFlowDataset(samples=1, use_kspace_noise=True, snr_range=(15.0, 15.0), seed=42)
lr_input, hr = dataset[0]

hr_speed = torch.linalg.vector_norm(hr, dim=0)
lr_speed = torch.linalg.vector_norm(lr_input[:3], dim=0)

z_lr = lr_input.shape[1] // 2
z_hr = hr.shape[1] // 2

# Create a beautiful, publication-ready grid
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=100)

# Apply a professional colormap ('magma' for speed, 'bone' for MRI magnitude)
im0 = axes[0].imshow(hr_speed[z_hr].cpu().numpy(), cmap='magma', origin='lower')
axes[0].set_title("Ground Truth HR Speed", fontsize=13, fontweight='semibold', pad=12)
axes[0].grid(False)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04).set_label("Speed (m/s)", fontsize=10)

im1 = axes[1].imshow(lr_speed[z_lr].cpu().numpy(), cmap='magma', origin='lower')
axes[1].set_title("K-Space Noisy LR Speed", fontsize=13, fontweight='semibold', pad=12)
axes[1].grid(False)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04).set_label("Speed (m/s)", fontsize=10)

im2 = axes[2].imshow(lr_input[3, z_lr].cpu().numpy(), cmap='bone', origin='lower')
axes[2].set_title("MRI Magnitude (Rayleigh Noise)", fontsize=13, fontweight='semibold', pad=12)
axes[2].grid(False)
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04).set_label("Intensity (a.u.)", fontsize=10)

for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Interactive 3D Ground Truth Flow Field

Let's visualize the ground-truth high-resolution 3D velocity vectors and translucent vessel boundaries using Plotly. This illustrates the target fluid domain before it is degraded by low-resolution MRI physics.

In [ ]:
fig_gt = make_interactive_3d_flow_figure(
    hr,
    wall_threshold=0.03,
    vector_threshold=0.10,
    vector_step=3,
    arrow_scale=0.15
)
fig_gt.show()

## 3. Define Train & Evaluation Loop

The evaluation loop tracks the performance of the models as well as the three mathematical upsampling baselines (Trilinear, Tricubic, and Sinc) on identical validation batches, calculating MAE, EPE (Endpoint Error), Peak Velocity Error, Net Flowrate Error, and L1 Divergence.

In [ ]:
@torch.no_grad()
def evaluate_all(model, loader):
    model.eval()
    totals = {
        'mae': 0.0, 'tri_mae': 0.0, 'cub_mae': 0.0, 'sinc_mae': 0.0,
        'epe': 0.0, 'tri_epe': 0.0, 'cub_epe': 0.0, 'sinc_epe': 0.0,
        'peak': 0.0, 'tri_peak': 0.0, 'cub_peak': 0.0, 'sinc_peak': 0.0,
        'flow': 0.0, 'tri_flow': 0.0, 'cub_flow': 0.0, 'sinc_flow': 0.0,
        'div': 0.0, 'tri_div': 0.0, 'cub_div': 0.0, 'sinc_div': 0.0
    }
    batches = 0
    for lr, hr in loader:
        lr, hr = lr.to(device), hr.to(device)
        pred = model(lr)
        
        # Mathematical upsamplings
        interp_tri = upsample_lr(lr, hr.shape[-1], mode="trilinear")
        interp_cub = upsample_lr(lr, hr.shape[-1], mode="tricubic")
        interp_snc = upsample_lr(lr, hr.shape[-1], mode="sinc")
        
        # MAE
        totals['mae'] += (pred - hr).abs().mean().item()
        totals['tri_mae'] += (interp_tri - hr).abs().mean().item()
        totals['cub_mae'] += (interp_cub - hr).abs().mean().item()
        totals['sinc_mae'] += (interp_snc - hr).abs().mean().item()
        
        # EPE
        totals['epe'] += endpoint_error(pred, hr).item()
        totals['tri_epe'] += endpoint_error(interp_tri, hr).item()
        totals['cub_epe'] += endpoint_error(interp_cub, hr).item()
        totals['sinc_epe'] += endpoint_error(interp_snc, hr).item()
        
        # Flow rate
        totals['flow'] += flow_rate_error(pred, hr).item()
        totals['tri_flow'] += flow_rate_error(interp_tri, hr).item()
        totals['cub_flow'] += flow_rate_error(interp_cub, hr).item()
        totals['sinc_flow'] += flow_rate_error(interp_snc, hr).item()
        
        # Peak velocity
        totals['peak'] += peak_velocity_error(pred, hr).item()
        totals['tri_peak'] += peak_velocity_error(interp_tri, hr).item()
        totals['cub_peak'] += peak_velocity_error(interp_cub, hr).item()
        totals['sinc_peak'] += peak_velocity_error(interp_snc, hr).item()
        
        # Divergence L1
        totals['div'] += divergence_l1(pred).item()
        totals['tri_div'] += divergence_l1(interp_tri).item()
        totals['cub_div'] += divergence_l1(interp_cub).item()
        totals['sinc_div'] += divergence_l1(interp_snc).item()
        
        batches += 1
    return {k: v / batches for k, v in totals.items()}

def train_model(upsample_mode, epochs=15, train_samples=1024, val_samples=128):
    train_dataset = SyntheticFlowDataset(train_samples, seed=10, use_kspace_noise=True, snr_range=(14.0, 17.0))
    val_dataset = SyntheticFlowDataset(val_samples, seed=10000, use_kspace_noise=True, snr_range=(14.0, 17.0))
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    
    model = FourDFlowNet(width=24, lr_blocks=4, hr_blocks=2, upsample_mode=upsample_mode).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type=='cuda')
    
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for lr, hr in train_loader:
            lr, hr = lr.to(device), hr.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda', enabled=device.type=='cuda'):
                pred = model(lr)
                loss = four_d_flow_loss(pred, hr, gradient_weight=1e-3)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        val_metrics = evaluate_all(model, val_loader)
        val_metrics['train_loss'] = epoch_loss
        val_metrics['epoch'] = epoch
        history.append(val_metrics)
        
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch}/{epochs} | Loss: {epoch_loss:.5f} | Val MAE: {val_metrics['mae']:.5f} | Peak Err: {100*val_metrics['peak']:.1f}% | Div L1: {val_metrics['div']:.5f}")
            
    return history, model

## 4. Execute Ablation Sweep

We run training loops for both upsampling choices inside the neural network:

In [ ]:
print("Training Paper Baseline (Trilinear Resize Layer)... ")
trilinear_history, trilinear_model = train_model("trilinear", epochs=15)

print("\nTraining Upgraded Architecture (Sub-pixel Convolution)... ")
subpixel_history, subpixel_model = train_model("subpixel", epochs=15)

## 5. Quantitative Results & Comparison

Compare the model outputs alongside all three mathematical interpolation baselines (Trilinear, Tricubic, and Sinc) across all medical metrics.

In [ ]:
t_final = trilinear_history[-1]
s_final = subpixel_history[-1]

summary = [
    {
        "Upsampling Method": "Trilinear (Math Baseline)",
        "Val MAE": f"{t_final['tri_mae']:.5f}",
        "EPE (Endpoint Err)": f"{t_final['tri_epe']:.5f}",
        "Peak Velocity Err%": f"{100*t_final['tri_peak']:.2f}%",
        "Net Flow Err%": f"{100*t_final['tri_flow']:.2f}%",
        "Divergence L1": f"{t_final['tri_div']:.5f}"
    },
    {
        "Upsampling Method": "Tricubic (Math Baseline)",
        "Val MAE": f"{t_final['cub_mae']:.5f}",
        "EPE (Endpoint Err)": f"{t_final['cub_epe']:.5f}",
        "Peak Velocity Err%": f"{100*t_final['cub_peak']:.2f}%",
        "Net Flow Err%": f"{100*t_final['cub_flow']:.2f}%",
        "Divergence L1": f"{t_final['cub_div']:.5f}"
    },
    {
        "Upsampling Method": "Sinc (Math Baseline)",
        "Val MAE": f"{t_final['sinc_mae']:.5f}",
        "EPE (Endpoint Err)": f"{t_final['sinc_epe']:.5f}",
        "Peak Velocity Err%": f"{100*t_final['sinc_peak']:.2f}%",
        "Net Flow Err%": f"{100*t_final['sinc_flow']:.2f}%",
        "Divergence L1": f"{t_final['sinc_div']:.5f}"
    },
    {
        "Upsampling Method": "Trilinear Model (Paper Architecture)",
        "Val MAE": f"{t_final['mae']:.5f}",
        "EPE (Endpoint Err)": f"{t_final['epe']:.5f}",
        "Peak Velocity Err%": f"{100*t_final['peak']:.2f}%",
        "Net Flow Err%": f"{100*t_final['flow']:.2f}%",
        "Divergence L1": f"{t_final['div']:.5f}"
    },
    {
        "Upsampling Method": "Sub-Pixel Model (Our Upgraded Architecture)",
        "Val MAE": f"{s_final['mae']:.5f}",
        "EPE (Endpoint Err)": f"{s_final['epe']:.5f}",
        "Peak Velocity Err%": f"{100*s_final['peak']:.2f}%",
        "Net Flow Err%": f"{100*s_final['flow']:.2f}%",
        "Divergence L1": f"{s_final['div']:.5f}"
    }
]
df = pd.DataFrame(summary)
print("\n=== Final Validation Performance Matrix ===")
print(df.to_markdown(index=False))


plt.figure(figsize=(11, 6), dpi=100)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

epochs = range(1, len(trilinear_history) + 1)

# Style model convergence lines
plt.plot(epochs, [h['mae'] for h in trilinear_history], 
         label='Trilinear Model (Paper)', color='#2B5C8F', linewidth=2.5, 
         marker='o', markersize=7, markeredgecolor='white', markeredgewidth=1.2)
plt.plot(epochs, [h['mae'] for h in subpixel_history], 
         label='Sub-Pixel Model (Ours)', color='#D95D39', linewidth=2.5, 
         marker='s', markersize=7, markeredgecolor='white', markeredgewidth=1.2)

# Style mathematical baseline lines
plt.axhline(y=t_final['tri_mae'], color='#7F8C8D', linestyle='--', linewidth=1.5, 
            label='Trilinear Interpolation')
plt.axhline(y=t_final['cub_mae'], color='#27AE60', linestyle='-.', linewidth=1.5, 
            label='Tricubic Interpolation')
plt.axhline(y=t_final['sinc_mae'], color='#8E44AD', linestyle=':', linewidth=1.5, 
            label='Sinc Interpolation')

# Titles and labels
plt.title('Ablation Study: Neural Super-Resolution vs. Mathematical Interpolation', 
          fontsize=14, fontweight='semibold', pad=15)
plt.xlabel('Epoch', fontsize=12, labelpad=8)
plt.ylabel('Validation MAE', fontsize=12, labelpad=8)

# Configure the grid
plt.grid(True, linestyle='--', alpha=0.5, color='#BDC3C7')

# Legend and ticks
plt.legend(frameon=True, facecolor='white', edgecolor='none', fontsize=11, loc='upper right')
plt.tick_params(axis='both', which='major', labelsize=11)

plt.tight_layout()
plt.show()

## 6. Masked Interactive 3D Flow Reconstruction

We use Plotly to visualize the final reconstructed 3D velocity vectors from the trained sub-pixel model. We mask the predictions using the ground-truth vessel domain to segment the flow region cleanly and remove background noise.

In [ ]:
ds_preview = SyntheticFlowDataset(samples=1, use_kspace_noise=True, seed=999)
lr, hr = ds_preview[0]
lr_batch = lr.unsqueeze(0).to(device)

subpixel_model.eval()
with torch.no_grad():
    pred_vel = subpixel_model(lr_batch).squeeze(0).cpu()

fig_pred = make_interactive_3d_flow_figure(
    pred_vel,
    wall_threshold=0.03,
    vector_threshold=0.10,
    vector_step=3,
    arrow_scale=0.15,
    ref_velocity=hr
)
fig_pred.show()